IT3100 Project: Hospital Readmission Predictor and Care Navigation Assistant

Project Overview

This notebook demonstrates the end-to-end pipeline of the IT3100 project, which combines:
- Machine Learning: Predicting hospital readmission risk for diabetic patients
- Generative AI: Providing Singapore-contextualized care navigation advice

The goal is to show how patient data flows from the ML model to the Gen AI assistant,
creating a unified clinical decision support tool.

Machine Learning Inference

This section loads the trained ML model and demonstrates how to calculate readmission risk
and Clinical Severity Score for a sample patient.

In [ ]:
# Load required libraries for ML inference
import joblib
import json
import pandas as pd
import numpy as np

# Load the trained model
model = joblib.load('outputs/readmission_model.joblib')
print("Model loaded successfully.")

# Load the expected feature columns
with open('outputs/feature_columns.json', 'r') as f:
    feature_columns = json.load(f)
print(f"Expected feature columns: {len(feature_columns)} features")

In [ ]:
# Create a sample high-risk patient profile
# This patient has prior admissions, high lab procedures, and specific symptoms

sample_patient = {
    # Core demographic features
    'age_numeric': 68,
    'is_elderly': 1,
    
    # Admission-related features
    'admission_type_id': 1,  # Emergency admission
    'discharge_disposition_id': 1,  # Home
    'admission_source_id': 4,  # ER
    'time_in_hospital': 7,
    
    # Healthcare utilization features
    'number_inpatient': 3,
    'number_outpatient': 8,
    'number_emergency': 2,
    'total_prior_admissions': 5,
    'emergency_ratio': 0.4,
    'inpatient_ratio': 0.3,
    'long_stay': 1,
    'er_admission': 1,
    'emergency_admission': 1,
    'not_home_discharge': 0,
    
    # Procedure and diagnosis features
    'num_procedures': 6,
    'num_lab_procedures': 45,
    'total_procedures': 51,
    'number_diagnoses': 9,
    'diabetes_diag_count': 2,
    'comorbidity_count': 7,
    'high_lab_utilization': 1,
    'high_diagnosis_count': 1,
    
    # Medication features
    'num_medications': 18,
    'total_medications': 18,
    'on_insulin': 1,
    'oral_medications': 1,
    'change_encoded': 1,
    'diabetesMed_encoded': 1,
    
    # Specific medication encodings (simplified - assuming active use)
    'metformin_encoded': 1, 'metformin_active': 1,
    'insulin_encoded': 1, 'insulin_active': 1,
    'glipizide_encoded': 1, 'glipizide_active': 1,
    'glyburide_encoded': 0, 'glyburide_active': 0,
    'pioglitazone_encoded': 0, 'pioglitazone_active': 0,
    
    # Other medication encodings (set to 0 for simplicity)
    'repaglinide_encoded': 0, 'repaglinide_active': 0,
    'nateglinide_encoded': 0, 'nateglinide_active': 0,
    'chlorpropamide_encoded': 0, 'chlorpropamide_active': 0,
    'glimepiride_encoded': 0, 'glimepiride_active': 0,
    'acetohexamide_encoded': 0, 'acetohexamide_active': 0,
    'tolbutamide_encoded': 0, 'tolbutamide_active': 0,
    'rosiglitazone_encoded': 0, 'rosiglitazone_active': 0,
    'acarbose_encoded': 0, 'acarbose_active': 0,
    'miglitol_encoded': 0, 'miglitol_active': 0,
    'troglitazone_encoded': 0, 'troglitazone_active': 0,
    'tolazamide_encoded': 0, 'tolazamide_active': 0,
    'examide_encoded': 0, 'examide_active': 0,
    'citoglipton_encoded': 0, 'citoglipton_active': 0,
    'glyburide-metformin_encoded': 0, 'glyburide-metformin_active': 0,
    'glipizide-metformin_encoded': 0, 'glipizide-metformin_active': 0,
    'glimepiride-pioglitazone_encoded': 0, 'glimepiride-pioglitazone_active': 0,
    'metformin-rosiglitazone_encoded': 0, 'metformin-rosiglitazone_active': 0,
    'metformin-pioglitazone_encoded': 0, 'metformin-pioglitazone_active': 0,
    
    # Interaction features
    'age_comorbidity_interaction': 476,  # 68 * 7
    'med_per_comorbidity': 2.57,  # 18 / 7
    'admissions_per_year': 0.5,  # 5 over 10 years
    'emerg_inpatient_combo': 5,  # emergency + inpatient count
    'insulin_complexity': 1,  # on insulin with multiple meds
    'diabetes_med_intensity': 3  # multiple diabetes medications
}

print("Sample patient profile created.")
print(f"Patient age: {sample_patient['age_numeric']}")
print(f"Prior admissions: {sample_patient['total_prior_admissions']}")
print(f"Lab procedures: {sample_patient['num_lab_procedures']}")
print(f"Comorbidities: {sample_patient['comorbidity_count']}")

In [ ]:
# Convert patient data to DataFrame with correct feature order
patient_df = pd.DataFrame([sample_patient])

# Ensure all expected columns are present and in correct order
for col in feature_columns:
    if col not in patient_df.columns:
        patient_df[col] = 0  # Default value for missing columns

# Reorder columns to match model's expected input
patient_df = patient_df[feature_columns]

# Make prediction
raw_probability = model.predict_proba(patient_df)[0][1]

# Calculate Clinical Severity Score (0-100 scale)
clinical_severity_score = int(raw_probability * 100)

# Classify risk category based on thresholds
if raw_probability < 0.20:
    risk_category = "Low"
    urgency_level = "Routine Monitoring"
elif raw_probability < 0.40:
    risk_category = "Moderate"
    urgency_level = "Increased Surveillance"
else:
    risk_category = "High"
    urgency_level = "Immediate Intervention"

print("="*50)
print("ML INFERENCE RESULTS")
print("="*50)
print(f"Raw Readmission Probability: {raw_probability:.4f}")
print(f"Clinical Severity Score: {clinical_severity_score} out of 100")
print(f"Risk Category: {risk_category}")
print(f"Urgency Level: {urgency_level}")
print("="*50)

Gen AI Care Navigation

This section uses the Gen AI assistant to generate personalized care advice
based on the patient's symptoms and the Clinical Severity Score from the ML model.

In [ ]:
# Import the Care Navigation Assistant from gen_ai.py
# Note: This requires GEMINI_API_KEY environment variable to be set

import sys
sys.path.append('.')

from gen_ai import CareNavigationAssistant

print("CareNavigationAssistant imported successfully.")
print("Note: Ensure GEMINI_API_KEY environment variable is set before generating advice.")

In [ ]:
# Define patient symptoms for the Gen AI context
patient_symptoms = [
    "fatigue",
    "frequent urination",
    "blurred vision",
    "slow wound healing",
    "numbness in feet"
]

print("Patient symptoms defined:")
for symptom in patient_symptoms:
    print(f"  - {symptom}")

In [ ]:
# Generate care navigation advice using Gen AI
# This integrates the ML-derived Clinical Severity Score with patient symptoms

try:
    # Initialize the assistant (requires GEMINI_API_KEY)
    assistant = CareNavigationAssistant()
    
    # Generate personalized advice
    advice = assistant.generate_advice(
        patient_symptoms=patient_symptoms,
        ml_risk_score=raw_probability,
        risk_category=risk_category,
        user_question="Please provide care navigation advice based on my current health status."
    )
    
    print("="*50)
    print("GEN AI CARE NAVIGATION ADVICE")
    print("="*50)
    print(advice)
    print("="*50)
    
except ValueError as e:
    print(f"API Key Error: {e}")
    print("\nTo run this section, set your Gemini API key:")
    print("  export GEMINI_API_KEY='your-api-key-here'")
    print("\nAlternatively, here is what the Gen AI would receive:")
    print(f"  Symptoms: {', '.join(patient_symptoms)}")
    print(f"  Clinical Severity Score: {clinical_severity_score}/100")
    print(f"  Risk Category: {risk_category}")
    print(f"  Urgency Level: {urgency_level}")

Pipeline Summary

How the ML model and Gen AI work together:

- The ML model provides a quantitative clinical severity baseline by analyzing patient features
  (age, prior admissions, lab procedures, comorbidities, medications) to calculate a
  readmission probability and Clinical Severity Score (0-100).

- The Gen AI assistant takes this score along with patient-reported symptoms to generate
  personalized, actionable care advice tailored to Singapore's healthcare system.

- The ML model grounds the Gen AI's qualitative recommendations with data-driven risk assessment,
  ensuring advice is appropriate for the patient's actual clinical severity level.

- Together, they create a unified clinical decision support tool that:
  - Identifies high-risk patients objectively
  - Provides context-aware care navigation (CHAS, Healthier SG, polyclinics)
  - Recommends appropriate urgency levels (routine vs immediate intervention)
  - Delivers patient-friendly explanations and next steps